# DS2002 · Capstone Design and Scoping

**Lecture — 2026-11-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Capstone kickoff — Game Day Pulse

Full brief: `DS2002_Capstone_Project_Brief.md`. Teams of 3–4, four weeks.

You are the data consultants for the **Cville Game Day Alliance**, the vendors working around Scott Stadium across the Fall 2026 home weekends. They want to know how to staff and stock. Five data sources, one of which is a SQLite database, plus live weather.

This is the midterm at larger scale with more sources and a longer runway. The teams that do well are the ones that treat the extra time as room to validate, not room to start late.

### What is different from the midterm

| Midterm | Capstone |
|---|---|
| One transactions file | Five sources, including a `.db` |
| Weather for a date range | Weather across several game weekends |
| Five questions | Five questions plus a staffing recommendation |
| Three weeks | Four weeks |
| Teams of 2–4 | Teams of 3–4 |

The genuinely new skill is joining across five sources without losing rows. Everything else you have done once already, which is why your carry-forward list from November 4th is the right place to start today.

### Profile all five sources

Same profiling function as the midterm. Run it on every source before anyone writes a line of cleaning code — you cannot scope work you have not looked at.

In [ ]:
import pandas as pd, sqlite3, os

def find_data(filename, folders=('data', '../data', '/kaggle/input', '/content')):
    for folder in folders:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return None

def profile(df, name):
    print(f'--- {name}: {len(df):,} rows x {len(df.columns)} cols ---')
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'null_pct': (100 * df.isnull().mean()).round(1),
        'distinct': df.nunique(),
    })
    print(summary)
    print('duplicate rows:', df.duplicated().sum())
    print()

SOURCES = ['gameday_orders.csv', 'vendor_locations.csv',
           'menu_catalog.csv', 'zone_capacity.csv']

frames = {}
for name in SOURCES:
    path = find_data(name)
    if path:
        frames[name] = pd.read_csv(path)
        profile(frames[name], name)
    else:
        print(f'{name}: not found -- upload the data/ folder\n')

### The database source

One source is SQLite rather than a CSV. Inspect it the way you learned in week 3: list the tables first, then read the schema of each.

In [ ]:
db = find_data('inventory_and_sales.db')
if db:
    conn = sqlite3.connect(db)
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", conn)
    print(tables)
    print()
    for t in tables['name']:
        info = pd.read_sql_query(f'PRAGMA table_info({t})', conn)
        n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t}', conn)['n'][0]
        print(f'{t}: {n:,} rows, columns =', info['name'].tolist())
else:
    print('inventory_and_sales.db not found -- upload the data/ folder')

### Check the join keys before you plan anything

This is the step that decides whether week 1 goes well. Five sources join on shared keys, and if those keys do not match formats you need to know today, not in week 3.

In [ ]:
if 'gameday_orders.csv' in frames and 'vendor_locations.csv' in frames:
    orders = frames['gameday_orders.csv']
    vendors = frames['vendor_locations.csv']

    order_ids = set(orders['vendor_id'].dropna().unique())
    vendor_ids = set(vendors['vendor_id'].dropna().unique())

    print('distinct vendor_id in orders: ', len(order_ids))
    print('distinct vendor_id in vendors:', len(vendor_ids))
    print('in orders but not vendors:', sorted(order_ids - vendor_ids)[:10])
    print('in vendors but not orders:', sorted(vendor_ids - order_ids)[:10])
    print()
    print('vendor_id unique in vendors?', vendors['vendor_id'].is_unique)

Read that output carefully. Ids in orders but not in vendors are orders you cannot attribute. A non-unique `vendor_id` in the vendor table will duplicate rows on every join. Both are week-1 work, and both are much cheaper to find today.

**TODO:** as a team, list at least six specific data-quality problems across the five sources, with counts where you have them.

_1._ ...

_2._ ...

_3._ ...

_4._ ...

_5._ ...

_6._ ...

### Team charter and four-week plan

Fill this in and commit it today. Note the week-by-week targets — a four-week project with no interim targets becomes a one-week project with three weeks of good intentions.

In [ ]:
team = {
    'name': 'TODO',
    'members': ['TODO', 'TODO', 'TODO'],
    'roles': {
        'cleaning + decision log': 'TODO',
        'weather API + joins': 'TODO',
        'SQL / database source': 'TODO',
        'analysis + charts': 'TODO',
    },
    'repo_url': 'TODO',
    'meeting_time': 'TODO',
    'plan': {
        'week 1 (Nov 16-20)': 'all 5 sources loading, cleaning started, one API call',
        'week 2 (Nov 23)':    'TODO -- short week, Thanksgiving',
        'week 3 (Nov 30)':    'TODO',
        'week 4 (Dec 7)':     'TODO -- presentation and submission',
    },
    'carry_forward_from_midterm': ['TODO', 'TODO', 'TODO'],
}

unfilled = [k for k, v in team.items() if 'TODO' in str(v)]
print('still unfilled:', unfilled if unfilled else 'nothing -- charter complete')

### Before you leave today

- All five sources profiled, including the database
- Join keys checked between orders and vendors
- Six specific problems listed with counts
- Charter committed, repository created, meeting time set

**This week's target:** every source loading, cleaning started, and one working weather call. Wednesday's studio is dedicated to exactly that, and Friday's lab grades it.